# Jyupiter Training Notebook

This notebook demonstrates how to train the `SvaraTransformer` model in the Raaga.AI repository using the existing training pipeline. It loads the dataset, builds the model, trains it, evaluates validation loss, and saves a checkpoint.

In [ ]:
# ── Colab / environment setup ────────────────────────────────────────────────
# Run this cell first. In Google Colab it:
#   1. Reads GITHUB_TOKEN from Colab Secrets (Key icon in the left sidebar)
#   2. Clones / updates the repository using that token
#   3. Changes cwd and installs Python dependencies
# Outside Colab this cell is a no-op.
#
# How to add the secret in Colab:
#   Left sidebar → Key icon (Secrets) → '+ Add new secret'
#   Name: GITHUB_TOKEN   Value: <your token>   toggle 'Notebook access' ON
#   Generate the token at: https://github.com/settings/tokens
#   Required scope: 'repo' (for private repos) or no scope (public repos)

import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

if IN_COLAB:
    # ── Read token from Colab Secrets (never stored in the notebook) ──────────
    GITHUB_TOKEN = ''
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or ''
    except Exception:
        GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')

    REPO_OWNER = 'Pariseashishbannu'
    REPO_NAME  = 'raagaai'
    REPO_DIR   = Path('/content') / REPO_NAME

    # Build clone URL — embed token for auth if provided
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('GitHub token found — using authenticated clone.')
    else:
        clone_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
        print('No GITHUB_TOKEN secret found — cloning without authentication (public repo only).')

    if not REPO_DIR.exists():
        print(f'Cloning {REPO_OWNER}/{REPO_NAME}...')
        subprocess.check_call(['git', 'clone', '--depth', '1', clone_url, str(REPO_DIR)])
    else:
        print('Repository already cloned — pulling latest changes...')
        # Update remote URL in case token changed
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', clone_url])
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    # Install Python requirements
    req_file = REPO_DIR / 'requirements.txt'
    if req_file.exists():
        print('Installing requirements...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)])
    else:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', '-q',
            'torch', 'requests', 'beautifulsoup4',
        ])

    print(f'Working directory: {os.getcwd()}')
else:
    print(f'Not in Colab — running from: {os.getcwd()}')


In [ ]:
import math
import time
import sys
from pathlib import Path

import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Locate the repository root using a repo-specific marker file.
# Checking bare src/ is not reliable — Linux has a system /src directory.
def _find_repo_root() -> Path:
    marker = Path('src') / 'ml' / 'model.py'   # unique to this repo
    cwd = Path.cwd()
    # Check cwd itself (Colab: os.chdir'd into the repo in the setup cell)
    if (cwd / marker).exists():
        return cwd
    # Walk upward (local dev: notebook opened from inside the repo)
    candidate = cwd.parent
    while candidate != candidate.parent:
        if (candidate / marker).exists():
            return candidate
        candidate = candidate.parent
    raise RuntimeError(
        f'Cannot find repo root (looked for {marker}). '
        f'cwd={cwd}. In Colab, make sure the setup cell ran successfully.'
    )

repo_root = _find_repo_root()

sys.path.insert(0, str(repo_root))

from src.ml.data.dataset import SvaraDataset
from src.ml.model import SvaraTransformer
from src.ml.tokenizer import SvaraTokenizer

pretrained_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer.pt'
notebook_checkpoint_path = repo_root / 'data' / 'models' / 'svara_transformer_notebook.pt'

print('Repository root:', repo_root)
print('Pretrained checkpoint:', pretrained_checkpoint_path)
print('Notebook checkpoint:', notebook_checkpoint_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)


: 

In [ ]:
# ── GPU & quantization auto-configuration ───────────────────────────────
# Runs automatically; configures MODEL_CFG, BATCH_SIZE, GRAD_ACCUM,
# AMP_DTYPE and USE_GRAD_CHECKPOINT for every GPU from a 4 GB laptop
# to a 24 GB workstation.  Change nothing unless you want to override.

import importlib, subprocess, sys

# bitsandbytes: 8-bit AdamW optimizer (4x less optimizer VRAM)
if importlib.util.find_spec('bitsandbytes') is None:
    print('Installing bitsandbytes ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])

import bitsandbytes as bnb

# ── Detect GPU ────────────────────────────────────────────────────────────
if device.type == 'cuda':
    _props   = torch.cuda.get_device_properties(device)
    _cc      = _props.major * 10 + _props.minor   # e.g. 86 for RTX 3050
    _vram_gb = _props.total_memory / 1e9
    GPU_NAME = _props.name

    # bf16: Ampere (cc>=80) and newer — no GradScaler needed, more stable
    USE_BF16  = _cc >= 80
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

    # TF32: faster Tensor Core matmul with no meaningful precision loss
    torch.backends.cuda.matmul.allow_tf32 = _cc >= 80
    torch.backends.cudnn.allow_tf32       = _cc >= 80
    torch.backends.cudnn.benchmark        = True
else:
    _vram_gb  = 0.0
    GPU_NAME  = 'CPU'
    USE_BF16  = False
    AMP_DTYPE = torch.float32

# ── Auto-size model and batch by VRAM ────────────────────────────────────
if _vram_gb <= 4:             # RTX 3050 laptop, GTX 1650 ...
    _cfg = (512,  8, 4,  8, 512)
    BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT = 32, 16, True
elif _vram_gb <= 8:           # RTX 3060, RTX 4060 ...
    _cfg = (512,  8, 4,  8, 512)
    BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT = 64,  8, False
elif _vram_gb <= 12:          # RTX 3080 10 GB, RTX 4070 ...
    _cfg = (768, 12, 6, 12, 512)
    BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT = 48, 11, False
elif _vram_gb <= 16:          # RTX 3080 16 GB, T4, RTX 4080 ...
    _cfg = (768, 12, 6, 12, 512)
    BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT = 64,  8, False
else:                         # RTX 3090, RTX 4090, A100 ...
    _cfg = (1024,16, 8, 16, 512)
    BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT = 128, 4, False

_embed, _heads, _kv, _layers, _seq = _cfg
MODEL_CFG = dict(
    vocab_size   = tokenizer.vocab_size,
    embed_dim    = _embed,
    num_heads    = _heads,
    num_kv_heads = _kv,
    num_layers   = _layers,
    max_seq_len  = _seq,
    dropout      = 0.1,
)

_est_params = _layers * 11 * _embed ** 2
print(f'GPU              : {GPU_NAME}')
print(f'VRAM             : {_vram_gb:.1f} GB')
print(f'Compute cap      : {_cc // 10}.{_cc % 10}')
print(f'Precision        : {"bf16" if USE_BF16 else "fp16"}')
print(f'Optimizer        : 8-bit AdamW (bitsandbytes {bnb.__version__})')
print(f'Model            : embed={_embed}  heads={_heads}  kv_heads={_kv}  layers={_layers}  seq={_seq}')
print(f'Est. parameters  : {_est_params / 1e6:.1f} M')
print(f'Batch / accum    : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} eff.')
print(f'Grad checkpoint  : {USE_GRAD_CHECKPOINT}')


## Build Dataset

Collect svara sequences from multiple sources and write them to
`data/training/svara_sequences.jsonl` so the training cells below can load them.

**Sources (in priority order):**

| # | Source | Notes |
|---|--------|-------|
| 1 | **Synthetic** — local raga definitions | Always enabled; no network needed |
| 2 | **karnatik.com** | Web scrape; toggle `SCRAPE_KARNATIK` |
| 3 | **shivkumar.org** | Web scrape; toggle `SCRAPE_SHIVKUMAR` |
| 4 | **Saraga dataset** | Set `SARAGA_DIR` to the extracted dataset path |
| 5 | **CompMusic Dunya API** | Set `DUNYA_TOKEN` (free registration required) |

Skip this section if `data/training/svara_sequences.jsonl` already exists and you
do not want to rebuild it.


In [ ]:
# Install web-scraping deps if missing (safe to run multiple times)
import importlib
import subprocess
import sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])


_ensure('requests')
_ensure('beautifulsoup4', 'bs4')
print('Dependencies ready.')


### Configuration

Edit the variables below to control which data sources are used.


### Saraga Carnatic Dataset

The [Saraga Carnatic 1.5](https://zenodo.org/record/4301060) dataset from CompMusic
contains 249 full concert recordings with svara phrase annotations in JSON format.

**Three ways to provide the dataset in Colab:**

| Option | When to use |
|--------|-------------|
| **A — Colab Secret** | You already uploaded the dataset to Google Drive. Add a secret `SARAGA_DIR` whose value is the Drive path, e.g. `/content/drive/MyDrive/saraga`. |
| **B — Auto-mount Drive** | Uncomment the `drive.mount` lines in the cell below; place the extracted dataset on your Drive. |
| **C — Download annotations** | The cell downloads only the lightweight JSON annotation files (~120 MB) from Zenodo — no audio. |

If none of these are set up, `SARAGA_DIR` stays `None` and this source is skipped
(synthetic + web scraping still run).


In [ ]:
import importlib, os, subprocess, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
SARAGA_DIR = None   # will be set below if a valid path is found

# ── Option A: Colab Secret 'SARAGA_DIR' ─────────────────────────────────
if IN_COLAB:
    try:
        from google.colab import userdata
        _secret = (userdata.get('SARAGA_DIR') or '').strip()
        if _secret and Path(_secret).exists():
            SARAGA_DIR = _secret
            print(f'[A] Saraga found via Colab Secret: {SARAGA_DIR}')
    except Exception:
        pass

# ── Option B: Google Drive mount (uncomment to use) ──────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# SARAGA_DIR = '/content/drive/MyDrive/saraga-carnatic-1.5'

# ── Option C: Download annotation-only archive from Zenodo (~120 MB) ─────
# Zenodo DOI 10.5281/zenodo.4301060 — annotations only (no audio)
_saraga_target = Path('/content/saraga-carnatic')
_annotation_url = (
    'https://zenodo.org/record/4301060/files/'
    'saraga1.5_carnatic.zip?download=1'
)

if SARAGA_DIR is None and IN_COLAB and not _saraga_target.exists():
    print('[C] Downloading Saraga Carnatic annotations from Zenodo ...')
    print('    (this is ~120 MB; audio files are excluded)')
    try:
        subprocess.check_call([
            'wget', '-q', '--show-progress',
            '-O', '/tmp/saraga.zip',
            _annotation_url,
        ])
        _saraga_target.mkdir(parents=True, exist_ok=True)
        subprocess.check_call([
            'unzip', '-q', '-n',
            '-d', str(_saraga_target),
            '/tmp/saraga.zip',
            '*.json',   # extract only JSON annotation files
        ])
        SARAGA_DIR = str(_saraga_target)
        print(f'[C] Saraga annotations extracted to: {SARAGA_DIR}')
    except Exception as e:
        print(f'[C] Download failed: {e}')
        print('    Falling back to synthetic + web sources.')

elif _saraga_target.exists() and SARAGA_DIR is None:
    SARAGA_DIR = str(_saraga_target)
    print(f'[C] Re-using previously downloaded Saraga data: {SARAGA_DIR}')

# ── Status ───────────────────────────────────────────────────────────────
if SARAGA_DIR and Path(SARAGA_DIR).exists():
    n_json = len(list(Path(SARAGA_DIR).rglob('*.json')))
    print(f'Saraga ready: {SARAGA_DIR}  ({n_json} JSON files found)')
else:
    SARAGA_DIR = None
    print('Saraga not available -- will be skipped in the scraper run.')


### CompMusic Dunya API

The [Dunya](https://dunya.compmusic.upf.edu) platform from CompMusic (Universitat Pompeu Fabra)
provides pitch contours and raga labels for hundreds of Carnatic recordings.
The `DunyaFetcher` converts raw Hz pitch contours into svara sequences automatically.

**How to get a free token:**
1. Register at `https://dunya.compmusic.upf.edu/user/register/`
2. Copy your token from `https://dunya.compmusic.upf.edu/user/profile/`
3. In Colab: click the **Key icon** in the left sidebar → **Add new secret**
   - Name: `DUNYA_TOKEN` · Value: paste your token · toggle **Notebook access ON**

If no token is provided this source is skipped.


In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
DUNYA_TOKEN = ''

# ── Read token from Colab Secrets (preferred) ────────────────────────────
if IN_COLAB:
    try:
        from google.colab import userdata
        DUNYA_TOKEN = (userdata.get('DUNYA_TOKEN') or '').strip()
    except Exception:
        pass

# ── Fall back to environment variable ────────────────────────────────────
if not DUNYA_TOKEN:
    DUNYA_TOKEN = os.environ.get('DUNYA_TOKEN', '').strip()

# ── Validate token with a lightweight API call ───────────────────────────
if DUNYA_TOKEN:
    try:
        import requests as _req
        _resp = _req.get(
            'https://dunya.compmusic.upf.edu/api/carnatic/recording',
            headers={'Authorization': f'Token {DUNYA_TOKEN}'},
            params={'limit': 1},
            timeout=15,
        )
        if _resp.status_code == 200:
            _total = _resp.json().get('count', '?')
            print(f'Dunya token valid. {_total} Carnatic recordings available.')
        elif _resp.status_code == 403:
            print('Dunya token rejected (403). Check your token and try again.')
            DUNYA_TOKEN = ''
        else:
            print(f'Dunya API returned {_resp.status_code} -- proceeding anyway.')
    except Exception as _e:
        print(f'Dunya connectivity check failed: {_e}')
        print('Proceeding -- will retry during scraping.')
else:
    print('No DUNYA_TOKEN set. Dunya source will be skipped.')
    print('Add DUNYA_TOKEN to Colab Secrets to enable it (see instructions above).')


### YouTube — CC-Licensed Carnatic Music

> **Copyright notice** 
> Downloading copyrighted YouTube content violates YouTube's Terms of Service (§5.B)
> and copyright law. This scraper **only processes videos whose license field
> is 'Creative Commons'** — enforced by inspecting yt-dlp metadata, not assumed.
> Do NOT change `CC_ONLY = True` below.

**Pipeline:**
```
YouTube search -> CC license check -> audio download (yt-dlp)
  -> pitch extraction (librosa pyin) -> tonic detection
  -> pitch-to-svara mapping -> raga label from title -> JSONL
```

**What this finds:** Carnatic concerts, veena recitals, vocal performances uploaded
by artists under Creative Commons licenses. It does **not** target Telugu film songs
(which are copyrighted and have no raga labels that would help training).

FFmpeg must be available for audio extraction (`apt-get install ffmpeg` in Colab).


In [ ]:
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# yt-dlp: download engine
if importlib.util.find_spec('yt_dlp') is None:
    print('Installing yt-dlp ...')
    _pip('yt-dlp')

# librosa: pitch extraction
if importlib.util.find_spec('librosa') is None:
    print('Installing librosa ...')
    _pip('librosa')

# FFmpeg: audio conversion (needed by yt-dlp post-processor)
import shutil
if shutil.which('ffmpeg') is None:
    print('Installing ffmpeg ...')
    subprocess.check_call(['apt-get', 'install', '-y', '-q', 'ffmpeg'])

import yt_dlp, librosa
print(f'yt-dlp  {yt_dlp.version.__version__}')
print(f'librosa {librosa.__version__}')
print(f'ffmpeg  {shutil.which("ffmpeg")}')
print('YouTube deps ready.')


In [ ]:
# ── YouTube scraper configuration ────────────────────────────────────────
SCRAPE_YOUTUBE  = True    # set False to skip this source
CC_ONLY         = True    # MUST stay True — do not download copyrighted content
YT_MAX_VIDEOS   = 30      # max CC-licensed videos to process
YT_SEGMENT_SECS = 30      # each video is split into N-second segments
YT_AUDIO_DIR    = '/tmp/raagaai_yt'  # temp folder for downloaded audio

# Search queries — target Carnatic classical music, NOT Telugu film songs
YT_QUERIES = [
    'carnatic classical music raga concert',
    'veena carnatic classical concert',
    'carnatic vocal concert full',
    'carnatic flute concert',
    'carnatic violin concert',
    'pallavi anupallavi carnatic',
    'karnatic music concert live',
]

print('YouTube config:')
print(f'  Enabled        : {SCRAPE_YOUTUBE}')
print(f'  CC only        : {CC_ONLY}  (enforced in scraper)')
print(f'  Max videos     : {YT_MAX_VIDEOS}')
print(f'  Segment length : {YT_SEGMENT_SECS}s')
print(f'  Audio cache    : {YT_AUDIO_DIR}')
print(f'  Queries        : {len(YT_QUERIES)}')


### SoundCloud — CC & Public-Domain Carnatic Tracks

> **Licence policy** — SoundCloud tracks carry a `license` field.
> This scraper accepts **Creative Commons and public-domain** licences and
> **rejects `all-rights-reserved`** tracks.  The same yt-dlp + librosa
> pipeline used for YouTube handles download and pitch extraction.

No extra token or account is needed — yt-dlp's `scsearch:` prefix queries
SoundCloud directly.  FFmpeg (already installed above) handles audio conversion.

**What this finds:** Carnatic concerts, veena and vocal recitals, and
instrumental ragas uploaded by artists under free licences on SoundCloud.


In [ ]:
# ── SoundCloud scraper configuration ─────────────────────────────────────
SCRAPE_SOUNDCLOUD = True   # set False to skip
SC_MAX_TRACKS     = 50     # max freely-licensed tracks to process
SC_SEGMENT_SECS   = 30     # seconds per extracted segment
SC_AUDIO_DIR      = '/tmp/raagaai_sc'

SC_QUERIES = [
    'carnatic classical music raga',
    'veena carnatic concert',
    'carnatic vocal concert',
    'karnatic music instrumental',
    'carnatic flute raga',
    'carnatic violin concert',
    'pallavi carnatic classical',
]

print('SoundCloud config:')
print(f'  Enabled        : {SCRAPE_SOUNDCLOUD}')
print(f'  Max tracks     : {SC_MAX_TRACKS}')
print(f'  Segment length : {SC_SEGMENT_SECS}s')
print(f'  Audio cache    : {SC_AUDIO_DIR}')
print(f'  Queries        : {len(SC_QUERIES)}')
print('Licence filter  : CC + public-domain only (all-rights-reserved skipped)')


In [ ]:
import os

# SARAGA_DIR and DUNYA_TOKEN were set by the setup cells above.
# Override here only if you want to force a specific value:
# SARAGA_DIR  = '/content/drive/MyDrive/saraga-carnatic-1.5'
# DUNYA_TOKEN = 'your-token'

# ── Web-scraper toggles ───────────────────────────────────────────────────
SCRAPE_KARNATIK  = True   # scrape karnatik.com
SCRAPE_SHIVKUMAR = True   # scrape shivkumar.org
MAX_WEB_PAGES    = 50     # max pages per web source

DUNYA_MAX_REC    = 100    # max recordings to fetch from Dunya

# ── Synthesis / augmentation ─────────────────────────────────────────────
SYNTHETIC_PER_RAGA = 200  # synthetic sequences generated per raga
AUGMENT            = True

# ── Output ───────────────────────────────────────────────────────────────
OUTPUT_PATH   = repo_root / 'data' / 'training' / 'svara_sequences.jsonl'
FORCE_REBUILD = False

print('=== Build Dataset config ===')
print(f'  Synthetic/raga   : {SYNTHETIC_PER_RAGA}')
print(f'  karnatik.com     : {SCRAPE_KARNATIK}  (max {MAX_WEB_PAGES} pages)')
print(f'  shivkumar.org    : {SCRAPE_SHIVKUMAR}  (max {MAX_WEB_PAGES} pages)')
print(f'  Saraga dir       : {SARAGA_DIR}')
print(f"  Dunya token      : {'SET (' + str(len(DUNYA_TOKEN)) + ' chars)' if DUNYA_TOKEN else 'not set'}")
print(f'  Dunya max recs   : {DUNYA_MAX_REC}')
print(f'  Augment          : {AUGMENT}')
print(f'  Output           : {OUTPUT_PATH}')
print(f'  Force rebuild    : {FORCE_REBUILD}')


### Run scrapers and save JSONL

This cell collects sequences from all enabled sources, applies augmentation,
shuffles, and writes the final dataset file.


In [ ]:
import json as _json
import logging
import random
from collections import Counter

from src.ml.data.scraper import (
    DunyaFetcher,
    KarnatikScraper,
    LocalRagaSynthesizer,
    SaragaScraper,
    ShivkumarScraper,
    augment_sequences,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

if OUTPUT_PATH.exists() and not FORCE_REBUILD:
    print(f'Dataset already exists at {OUTPUT_PATH} — skipping build.')
    print('Set FORCE_REBUILD = True in the config cell to regenerate it.')
else:
    all_sequences: list[dict] = []

    # 1. Synthetic sequences (always)
    print('[1/7] Generating synthetic sequences from local raga definitions...')
    synth = LocalRagaSynthesizer(str(repo_root / 'data' / 'ragas'))
    synthetic = synth.synthesize(sequences_per_raga=SYNTHETIC_PER_RAGA)
    all_sequences.extend(synthetic)
    print(f'      {len(synthetic):,} synthetic sequences')

    # 2. karnatik.com
    if SCRAPE_KARNATIK:
        print(f'[2/7] Scraping karnatik.com (max {MAX_WEB_PAGES} pages)...')
        karnatik_seqs = KarnatikScraper().scrape(max_pages=MAX_WEB_PAGES)
        all_sequences.extend(karnatik_seqs)
        print(f'      {len(karnatik_seqs):,} sequences from karnatik.com')
    else:
        print('[2/7] karnatik.com skipped (SCRAPE_KARNATIK=False)')

    # 3. shivkumar.org
    if SCRAPE_SHIVKUMAR:
        print(f'[3/7] Scraping shivkumar.org (max {MAX_WEB_PAGES} pages)...')
        shiv_seqs = ShivkumarScraper().scrape(max_pages=MAX_WEB_PAGES)
        all_sequences.extend(shiv_seqs)
        print(f'      {len(shiv_seqs):,} sequences from shivkumar.org')
    else:
        print('[3/7] shivkumar.org skipped (SCRAPE_SHIVKUMAR=False)')

    # 4. Saraga dataset
    if SARAGA_DIR:
        print(f'[4/7] Parsing Saraga dataset from {SARAGA_DIR}...')
        saraga_seqs = SaragaScraper().scrape(SARAGA_DIR)
        all_sequences.extend(saraga_seqs)
        print(f'      {len(saraga_seqs):,} sequences from Saraga')
    else:
        print('[4/7] Saraga skipped (SARAGA_DIR not set)')

    # 5. CompMusic Dunya API
    if DUNYA_TOKEN:
        print(f'[5/7] Fetching from Dunya (max {DUNYA_MAX_REC} recordings)...')
        dunya_seqs = DunyaFetcher(DUNYA_TOKEN).fetch(max_recordings=DUNYA_MAX_REC)
        all_sequences.extend(dunya_seqs)
        print(f'      {len(dunya_seqs):,} sequences from Dunya')
    else:
        print('[5/7] Dunya skipped (DUNYA_TOKEN not set)')

    if SCRAPE_YOUTUBE:
        print(f'[6/7] Scraping CC-licensed YouTube videos (max {YT_MAX_VIDEOS})...')
        from src.ml.data.scraper import YouTubeScraper
        yt_seqs = YouTubeScraper(output_dir=YT_AUDIO_DIR).scrape(
            max_videos=YT_MAX_VIDEOS,
            segment_secs=YT_SEGMENT_SECS,
            queries=YT_QUERIES,
        )
        all_sequences.extend(yt_seqs)
        print(f'      {len(yt_seqs):,} sequences from YouTube (CC-licensed only)')
    else:
        print('[6/7] YouTube skipped (SCRAPE_YOUTUBE=False)')

    if SCRAPE_SOUNDCLOUD:
        print(f'[7/7] Scraping SoundCloud (max {SC_MAX_TRACKS} freely-licensed tracks)...')
        from src.ml.data.scraper import SoundCloudScraper
        sc_seqs = SoundCloudScraper(output_dir=SC_AUDIO_DIR).scrape(
            max_tracks=SC_MAX_TRACKS,
            segment_secs=SC_SEGMENT_SECS,
            queries=SC_QUERIES,
        )
        all_sequences.extend(sc_seqs)
        print(f'      {len(sc_seqs):,} sequences from SoundCloud (CC / public-domain)')
    else:
        print('[7/7] SoundCloud skipped (SCRAPE_SOUNDCLOUD=False)')

    if not all_sequences:
        raise RuntimeError('No sequences collected — check your data source configuration.')

    # Augment
    if AUGMENT:
        print(f'\nAugmenting {len(all_sequences):,} sequences...')
        all_sequences = augment_sequences(all_sequences)
        print(f'  -> {len(all_sequences):,} sequences after augmentation')

    # Shuffle and save
    random.shuffle(all_sequences)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as _f:
        for seq in all_sequences:
            _f.write(_json.dumps(seq, ensure_ascii=False) + '\n')

    # Stats
    raga_counts = Counter(s['raga_id'] for s in all_sequences)
    section_counts = Counter(s['section'] for s in all_sequences)
    total_svaras = sum(len(s['svaras']) for s in all_sequences)

    print(f'\n=== Dataset saved to {OUTPUT_PATH} ===')
    print(f'  Total sequences : {len(all_sequences):,}')
    print(f'  Unique ragas    : {len(raga_counts)}')
    print(f'  Sections        : {dict(section_counts)}')
    print(f'  Total svaras    : {total_svaras:,}')
    print(f'  Avg seq length  : {total_svaras / len(all_sequences):.1f}')
    print(f'\nTop 10 ragas by sequence count:')
    for raga, cnt in raga_counts.most_common(10):
        print(f'    {raga:<30} {cnt:,}')

## Load Dataset

The Raaga.AI training data is expected at `data/training/svara_sequences.jsonl`. This notebook loads that file and inspects the dataset size.

In [ ]:
data_path = repo_root / 'data' / 'training' / 'svara_sequences.jsonl'
print('Training data path:', data_path)
print('Exists:', data_path.exists())

if not data_path.exists():
    raise FileNotFoundError(
        f'Training data not found at {data_path}\n'
        'Run the \'Build Dataset\' section at the top of this notebook first, '
        'then re-run this cell.'
    )

tokenizer = SvaraTokenizer()
dataset = SvaraDataset(str(data_path), tokenizer, max_length=256)
print('Dataset size:', len(dataset))
print('Vocabulary size:', tokenizer.vocab_size)

sample = dataset[0] if len(dataset) > 0 else None
print('Sample keys:', list(sample.keys()) if sample else None)
if sample:
    print('Input IDs length:', sample['input_ids'].shape)
    print('Labels length:', sample['labels'].shape)


## Preprocess Data

Create train/validation splits and PyTorch data loaders for batch training. This step uses the same sequence format and tokenizer as the repo's training script.

In [ ]:
# max_seq_len, BATCH_SIZE, GRAD_ACCUM set by the GPU config cell
max_seq_len = MODEL_CFG['max_seq_len']
val_split   = 0.05

from src.ml.data.dataset import SvaraDataset
dataset = SvaraDataset(str(data_path), tokenizer, max_length=max_seq_len)

val_size   = max(1, int(len(dataset) * val_split))
train_size = len(dataset) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size]
)

_nw = min(4, __import__('os').cpu_count() or 1)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size         = BATCH_SIZE,
    shuffle            = True,
    num_workers        = _nw,
    pin_memory         = (device.type == 'cuda'),
    prefetch_factor    = 2 if _nw > 0 else None,
    persistent_workers = (_nw > 0),
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size         = BATCH_SIZE * 2,
    shuffle            = False,
    num_workers        = _nw,
    pin_memory         = (device.type == 'cuda'),
    prefetch_factor    = 2 if _nw > 0 else None,
    persistent_workers = (_nw > 0),
)

print(f'Sequence length : {max_seq_len}')
print(f'Train samples   : {len(train_dataset):,}')
print(f'Val samples     : {len(val_dataset):,}')
print(f'Batch size      : {BATCH_SIZE}  '
      f'(effective {BATCH_SIZE * GRAD_ACCUM} with grad-accum x{GRAD_ACCUM})')
print(f'DataLoader workers: {_nw}')


## Define Model Architecture

Instantiate the `SvaraTransformer` from `src/ml/model.py` with the same architecture used by the existing training script.

In [ ]:
import importlib, subprocess as _sp, sys as _sys

if importlib.util.find_spec('tqdm') is None:
    _sp.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'])
from tqdm.auto import tqdm

# MODEL_CFG, BATCH_SIZE, GRAD_ACCUM, USE_GRAD_CHECKPOINT, AMP_DTYPE
# are all set by the GPU config cell above.
model = SvaraTransformer(**MODEL_CFG).to(device)

# ── Gradient checkpointing ────────────────────────────────────────────────
# Recomputes activations during backward instead of storing them.
# Cuts activation memory ~60 % at the cost of ~33 % more compute.
if USE_GRAD_CHECKPOINT:
    from torch.utils.checkpoint import checkpoint as _ckpt
    for _blk in model.layers:
        _orig = _blk.forward
        def _wrap(x, cos, sin, mask, _f=_orig):
            return _ckpt(_f, x, cos, sin, mask, use_reentrant=False)
        _blk.forward = _wrap
    print('Gradient checkpointing: ON')

# ── torch.compile (PyTorch >= 2.0, ~20 % speed-up) ───────────────────────
if hasattr(torch, 'compile'):
    model = torch.compile(model)
    print('torch.compile(): applied')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters : {n_params:,}  ({n_params / 1e6:.1f} M)')
if device.type == 'cuda':
    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated(device) / 1e9
    total = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f'VRAM used  : {alloc:.2f} GB / {total:.1f} GB')


## Compile Model

Set up optimizer, scheduler, and training hyperparameters. This notebook mirrors the repo's `scripts/train_model.py` defaults.

If a pretrained checkpoint exists, it is loaded and training is skipped; otherwise the notebook will train for the full number of epochs on the available dataset.


In [ ]:
import bitsandbytes as bnb
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import GradScaler

learning_rate = 3e-4
weight_decay  = 0.1
grad_clip     = 1.0
full_epochs   = 100
warmup_epochs = 5

# ── Load checkpoint if available ─────────────────────────────────────────
if notebook_checkpoint_path.exists():
    ckpt_src = notebook_checkpoint_path
elif pretrained_checkpoint_path.exists():
    ckpt_src = pretrained_checkpoint_path
else:
    ckpt_src = None

if ckpt_src:
    epochs = 0
    _ckpt  = torch.load(ckpt_src, map_location=device)
    model.load_state_dict(_ckpt['model_state_dict'])
    print('Loaded checkpoint:', ckpt_src)
else:
    epochs = full_epochs
    print('No checkpoint — training from scratch for', epochs, 'epochs.')

# ── 8-bit AdamW optimizer (bitsandbytes) ─────────────────────────────────
# Quantises m/v states to int8 -> 4x smaller optimizer memory.
# fp32 master weights are maintained internally for accurate updates.
decay_p    = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() >= 2]
no_decay_p = [p for n, p in model.named_parameters() if p.requires_grad and p.dim() <  2]

try:
    optimizer = bnb.optim.AdamW8bit(
        [{'params': decay_p,    'weight_decay': weight_decay},
         {'params': no_decay_p, 'weight_decay': 0.0}],
        lr=learning_rate, betas=(0.9, 0.95), eps=1e-8,
    )
    print(f'Optimizer: 8-bit AdamW (bitsandbytes)  '
          f'optimizer VRAM ~{sum(p.numel() for p in model.parameters())*2/1e6:.0f} MB')
except Exception as _e:
    from torch.optim import AdamW
    optimizer = AdamW(
        [{'params': decay_p,    'weight_decay': weight_decay},
         {'params': no_decay_p, 'weight_decay': 0.0}],
        lr=learning_rate, betas=(0.9, 0.95), eps=1e-8,
    )
    print(f'8-bit AdamW unavailable ({_e}) — using standard AdamW.')

# ── Scheduler: linear warmup then cosine decay ───────────────────────────
warmup_sched = LinearLR(optimizer, start_factor=0.01, end_factor=1.0,
                         total_iters=warmup_epochs)
cosine_sched = CosineAnnealingLR(optimizer, T_max=max(1, epochs - warmup_epochs),
                                  eta_min=learning_rate * 0.1)
scheduler    = SequentialLR(optimizer, [warmup_sched, cosine_sched],
                              milestones=[warmup_epochs])

# ── GradScaler — only needed for fp16 (bf16 has a wide enough exponent range) ─
USE_AMP    = device.type == 'cuda'
USE_SCALER = USE_AMP and not USE_BF16
scaler     = GradScaler(enabled=USE_SCALER)

print(f'Precision  : {"bf16 (no GradScaler)" if USE_BF16 else "fp16 + GradScaler"}')
print(f'LR         : {learning_rate}  epochs: {epochs}  warmup: {warmup_epochs}')
print(f'Grad clip  : {grad_clip}  accum: {GRAD_ACCUM}')


## Train Model

Run the training loop. The notebook logs training and validation loss per epoch and saves the best checkpoint.

In [ ]:
from torch.cuda.amp import autocast


def vram_str() -> str:
    if device.type != 'cuda':
        return ''
    alloc = torch.cuda.memory_allocated(device) / 1e9
    resv  = torch.cuda.memory_reserved(device)  / 1e9
    return f'  VRAM {alloc:.1f}/{resv:.1f} GB'


def train_one_epoch(model, loader, optimizer, scaler, device, grad_accum):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(loader, desc='  train', leave=False, dynamic_ncols=True)
    for step, batch in enumerate(pbar, 1):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        labels    = batch['labels'].to(device, non_blocking=True)

        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = model(input_ids, labels=labels)['loss'] / grad_accum

        if USE_SCALER:
            scaler.scale(loss).backward()
        else:
            loss.backward()                 # bf16: no scaler needed

        total_loss += loss.item() * grad_accum

        if step % grad_accum == 0 or step == len(loader):
            if USE_SCALER:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=f'{loss.item() * grad_accum:.4f}', vram=vram_str())
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, steps = 0.0, 0
    for batch in tqdm(loader, desc='  val  ', leave=False, dynamic_ncols=True):
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        labels    = batch['labels'].to(device, non_blocking=True)
        with autocast(dtype=AMP_DTYPE, enabled=USE_AMP):
            total += model(input_ids, labels=labels)['loss'].item()
        steps += 1
    return total / max(steps, 1)


output_dir = repo_root / 'data' / 'models'
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = notebook_checkpoint_path

best_val_loss, patience, patience_counter = float('inf'), 15, 0
start_time = time.time()

if epochs == 0:
    print('Pretrained checkpoint loaded — skipping training.')
else:
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device, GRAD_ACCUM)
        val_loss   = evaluate(model, val_loader, device)
        scheduler.step()

        elapsed = time.time() - start_time
        lr_now  = optimizer.param_groups[0]['lr']
        vram_info = ''
        if device.type == 'cuda':
            peak_gb   = torch.cuda.max_memory_allocated(device) / 1e9
            vram_info = f'  peak {peak_gb:.1f} GB'
            torch.cuda.reset_peak_memory_stats(device)

        print(f'Epoch {epoch:>3}/{epochs}'
              f'  train {train_loss:.4f}  val {val_loss:.4f}'
              f'  lr {lr_now:.2e}  {elapsed/60:.1f} min{vram_info}')

        if val_loss < best_val_loss:
            best_val_loss, patience_counter = val_loss, 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'config': {**MODEL_CFG, 'dropout': 0.0},
                'tokenizer_vocab': tokenizer.token_to_id,
                'epoch': epoch, 'val_loss': best_val_loss,
                'precision': str(AMP_DTYPE),
            }, checkpoint_path)
            print(f'  [saved] val={best_val_loss:.4f} -> {checkpoint_path.name}')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch}.')
                break

    print(f'Done.  Best val: {best_val_loss:.4f}  Time: {(time.time()-start_time)/60:.1f} min')


## Evaluate Model

Run a final evaluation on the held-out validation split to confirm the checkpoint quality.

In [ ]:
final_val_loss = evaluate(model, val_loader, device)
print('Final validation loss:', final_val_loss)


## Post-Training Quantization

Converts the trained model to **int8** for fast CPU inference with ~4x smaller file size.
Uses PyTorch dynamic quantization — no extra calibration data needed.


In [ ]:
# ── Post-training dynamic int8 quantization ─────────────────────────────
# Converts all nn.Linear layers to int8 for CPU inference.
# No retraining needed — PyTorch calibrates from the weights directly.
# Result: ~4x smaller model file, ~2-3x faster on CPU, tiny accuracy drop.

import copy

# Load the best checkpoint into a fresh CPU model
_ckpt_data = torch.load(checkpoint_path, map_location='cpu')
_cfg       = _ckpt_data['config']

model_fp32 = SvaraTransformer(
    vocab_size   = _cfg['vocab_size'],
    embed_dim    = _cfg['embed_dim'],
    num_heads    = _cfg['num_heads'],
    num_kv_heads = _cfg.get('num_kv_heads', _cfg['num_heads'] // 2),
    num_layers   = _cfg['num_layers'],
    max_seq_len  = _cfg['max_seq_len'],
    dropout      = 0.0,
).cpu()
model_fp32.load_state_dict(_ckpt_data['model_state_dict'], strict=False)
model_fp32.eval()

# Dynamic quantisation — weights quantised to int8 on-the-fly at runtime
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    qconfig_spec={torch.nn.Linear},
    dtype=torch.qint8,
)

# Save
quant_path = output_dir / 'svara_transformer_int8.pt'
torch.save({'model': model_int8, 'config': _cfg,
            'tokenizer_vocab': _ckpt_data['tokenizer_vocab']}, quant_path)

# Compare sizes and a quick speed test
import os, time as _t
fp32_mb = os.path.getsize(checkpoint_path) / 1e6
int8_mb = os.path.getsize(quant_path)      / 1e6

_dummy  = torch.randint(0, _cfg['vocab_size'], (1, 16))
_runs   = 20

_t0 = _t.perf_counter()
for _ in range(_runs):
    with torch.no_grad(): model_fp32(_dummy)
fp32_ms = (_t.perf_counter() - _t0) / _runs * 1000

_t0 = _t.perf_counter()
for _ in range(_runs):
    with torch.no_grad(): model_int8(_dummy)
int8_ms = (_t.perf_counter() - _t0) / _runs * 1000

print(f'fp32 checkpoint : {fp32_mb:.1f} MB  |  {fp32_ms:.1f} ms/forward')
print(f'int8 model      : {int8_mb:.1f} MB  |  {int8_ms:.1f} ms/forward')
print(f'Size reduction  : {fp32_mb / int8_mb:.1f}x smaller')
print(f'Speed-up (CPU)  : {fp32_ms / int8_ms:.1f}x faster')
print(f'Saved to        : {quant_path}')


## Save Trained Model

The notebook saves the trained checkpoint and generates a sample melody using the same tokenizer.

In [ ]:
print('Checkpoint saved at:', checkpoint_path)

model.eval()
prefix_tokens = tokenizer.encode_conditioning('shankarabharanam', 'adi', 'pallavi', 'madhyama')
prefix_tensor = torch.tensor([prefix_tokens], device=device)
generated = model.generate(prefix_tensor, max_new_tokens=32, temperature=0.8, top_k=15)
decoded = tokenizer.decode(generated[0].cpu().tolist())
print('Generated sequence:')
print(' '.join(decoded))
